# Step 5 - landscape and perturbation

Stochastic relaxation to a quasi-potential landscape, then kick-and-release
for every gene pair.

In [ ]:
import os
import sys

NETDESDUO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "neutrophil_data"))
sys.path.insert(0, NETDESDUO_ROOT)
import NetDesDuo
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import random
import importlib
import math
import joblib
from sklearn.metrics import mean_squared_error
importlib.reload(NetDesDuo)
import math
import matplotlib.pyplot as plt
import itertools
from sklearn.decomposition import PCA

# fitting run tree - not committed, edit for your setup
WORK_DIR = "/projects/lulab/alex/neutrophil_case/network_optimization"
os.chdir(WORK_DIR)

## Fitted models for both conditions

In [ ]:
# getting naive data
tf11 = pd.read_csv('naive_results/naive_expression_data.csv')
genes = pd.read_csv('naive_results/naive_genes_expressed.csv')
genes = genes['x'].to_list()

tf11 = tf11.transpose()
tf11 = tf11[1:100000]
tf11.index = genes

# restore the average pseudotime values to before log-ing
naive_tf12expression_all = tf11.apply(lambda x: (math.e**x - 1))
naive_tf12expression_all[naive_tf12expression_all <= 0.001] = 0.001

# scale expression
naive_tf12_scaled_all = naive_tf12expression_all.div(
    naive_tf12expression_all.max(axis=1),
    axis=0
)

naive_tf12expression_logtarget_all = naive_tf12_scaled_all.apply(lambda x: np.log2(x))
naive_tf12expression_logtarget_all = naive_tf12expression_logtarget_all.dropna()
naive_tf12expression_all = naive_tf12expression_all.dropna()

network0 = pd.read_csv('naive_results/naive_initial_network.csv')
network1 = network0[["Source", "Target", "Interaction"]]
naive_gene_list = list(pd.unique(network1['Target']))
network1.columns = ['Source', 'Target', 'Interaction']

naive_expression = naive_tf12expression_all.loc[naive_gene_list]
naive_expression_log = naive_tf12expression_logtarget_all.loc[naive_gene_list]

naive_network = pd.read_csv("naive_results/naive_combined_signed.csv")
naive_res_final = joblib.load("naive_results/naive_networks/n_0.5_k_1.5/naive_res_final_signed.joblib")
naive_ob_newall = joblib.load("naive_results/naive_networks/n_0.5_k_1.5/naive_ob_newall_signed.joblib")

In [ ]:
# getting cancer data
tf11 = pd.read_csv('cancer_results/cancer_expression_data.csv')
genes = pd.read_csv('cancer_results/cancer_genes_expressed.csv')
genes = genes['x'].to_list()

tf11 = tf11.transpose()
tf11 = tf11[1:100000]
tf11.index = genes

# restore the average pseudotime values to before log-ing
cancer_tf12expression_all = tf11.apply(lambda x: (math.e**x - 1))
cancer_tf12expression_all[cancer_tf12expression_all <= 0.001] = 0.001

# scale expression
cancer_tf12_scaled_all = cancer_tf12expression_all.div(
    cancer_tf12expression_all.max(axis=1),
    axis=0
)

cancer_tf12expression_logtarget_all = cancer_tf12_scaled_all.apply(lambda x: np.log2(x))
cancer_tf12expression_logtarget_all = cancer_tf12expression_logtarget_all.dropna()
cancer_tf12expression_all = cancer_tf12expression_all.dropna()

network0 = pd.read_csv('cancer_results/cancer_initial_network.csv')
network1 = network0[["Source", "Target", "Interaction"]]
cancer_gene_list = list(pd.unique(network1['Target']))
network1.columns = ['Source', 'Target', 'Interaction']

cancer_expression = cancer_tf12expression_all.loc[cancer_gene_list]
cancer_expression_log = cancer_tf12expression_logtarget_all.loc[cancer_gene_list]

cancer_network = pd.read_csv("cancer_results/cancer_combined_signed.csv")
cancer_res_final = joblib.load("cancer_results/cancer_networks/n_0.5_k_1.5/cancer_res_final_signed.joblib")
cancer_ob_newall = joblib.load("cancer_results/cancer_networks/n_0.5_k_1.5/cancer_ob_newall_signed.joblib")
pseudotime_pick = pd.read_csv('naive_results/pseudotime_pick.csv')[['x']]

naive_specific_genes = ["Stat4", "Ets1", "Rel"]
cancer_specific_genes = ["Sp3", "Relb", "Jund"]
overlap_genes = list(set(cancer_gene_list) & set(naive_gene_list))

## Landscape: baseline steady-state clouds

In [ ]:
# -----------------------------
# Naive baseline steady-state PCA cloud
# -----------------------------

# gene positions for naive network
gene_to_i = {g: i for i, g in enumerate(naive_gene_list)}

gene_position2 = [
    [gene_to_i[r] for r in regs]
    for regs in naive_ob_newall
]

# scaled expression: genes x pseudotime
naive_scaled = naive_expression.div(naive_expression.max(axis=1), axis=0)

# noise scale vector
mean_exp2 = naive_scaled.mean(axis=1).values.astype(float)

start_end = [
    naive_scaled.iloc[:, 0].values.astype(float),
    naive_scaled.iloc[:, -1].values.astype(float)
]
n_runs = 5000
n_genes = len(naive_gene_list)

results = np.zeros((n_genes, n_runs))

low_vec = start_end[0]
high_vec = start_end[1]

lo = np.minimum(low_vec, high_vec)
hi = np.maximum(low_vec, high_vec)

np.random.seed(1)

for i in range(n_runs):
    rand_exp = np.random.uniform(lo, hi)

    df_state = NetDesDuo.dynamic_state(
        dt=1,
        t_tot=2000,
        exp_array=rand_exp,
        parameters_all=naive_res_final,
        gene_position=gene_position2,
        e_mean=mean_exp2,
        save_num=201
    )

    results[:, i] = df_state.iloc[:, -1].values


naive_baseline_cloud = pd.DataFrame(
    results,
    index=naive_gene_list,
    columns=[f"run_{j+1}" for j in range(n_runs)]
)
naive_baseline_cloud.to_csv("naive_perturbations/naive_baseline_cloud.csv")

In [ ]:
# -----------------------------
# Naive baseline cloud + full real pseudotime trajectory
# -----------------------------
naive_baseline_cloud = pd.read_csv("naive_perturbations/naive_baseline_cloud.csv", index_col = 0)

# baseline cloud: runs x genes
X_cloud = naive_baseline_cloud.T.values

# fit PCA on the simulated baseline cloud
pca_model = PCA(n_components=2, random_state=0)
pcs_cloud = pca_model.fit_transform(X_cloud)

explained = pca_model.explained_variance_ratio_ * 100

pcs_cloud_df = pd.DataFrame(
    pcs_cloud,
    index=naive_baseline_cloud.columns,
    columns=["PC1", "PC2"]
)

# real pseudotime data in same scaled expression space
naive_scaled = naive_expression.div(naive_expression.max(axis=1), axis=0)
naive_scaled = naive_scaled.loc[naive_gene_list]

# pseudotime points: timepoints x genes
X_real = naive_scaled.T.values

# project real pseudotime trajectory into the baseline PCA axes
pcs_real = pca_model.transform(X_real)

pcs_real_df = pd.DataFrame(
    pcs_real,
    columns=["PC1", "PC2"]
)

pcs_real_df["pseudotime_index"] = np.arange(len(pcs_real_df))

# plot
plt.figure(figsize=(6, 5))

plt.scatter(
    pcs_cloud_df["PC1"],
    pcs_cloud_df["PC2"],
    s=12,
    alpha=0.35,
    label="Simulated States"
)


#plt.scatter(
#    pcs_real_df["PC1"],
#    pcs_real_df["PC2"],
#    c=pcs_real_df["pseudotime_index"],
#    s=25,
#    cmap="viridis",
#    label="Real pseudotime points"
#)

plt.scatter(
    pcs_real_df["PC1"].iloc[0],
    pcs_real_df["PC2"].iloc[0],
    s=120,
    marker="o",
    edgecolor="black",
    linewidth=1.2,
    label="Observed Start"
)

plt.scatter(
    pcs_real_df["PC1"].iloc[-1],
    pcs_real_df["PC2"].iloc[-1],
    s=120,
    marker="s",
    edgecolor="black",
    linewidth=1.2,
    label="Observed End"
)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.xlabel(f"PC1 ({explained[0]:.1f}%)", fontsize = 14)
plt.ylabel(f"PC2 ({explained[1]:.1f}%)", fontsize = 14)
#plt.colorbar(label="Pseudotime index")
plt.legend(fontsize=10)
plt.tight_layout()

plt.savefig('naive_results/naive_stochastic_simulation.png', dpi = 600)
plt.show()

In [ ]:
# -----------------------------
# Cancer baseline steady-state PCA cloud
# -----------------------------

# gene positions for cancer network
gene_to_i = {g: i for i, g in enumerate(cancer_gene_list)}

gene_position2 = [
    [gene_to_i[r] for r in regs]
    for regs in cancer_ob_newall
]

# scaled expression: genes x pseudotime
cancer_scaled = cancer_expression.div(cancer_expression.max(axis=1), axis=0)

# noise scale vector
mean_exp2 = cancer_scaled.mean(axis=1).values.astype(float)

#find max vs min
cancer_start_end = [
    cancer_scaled.min(axis=1).values.astype(float),
    cancer_scaled.max(axis=1).values.astype(float),
]

n_runs = 5000
n_genes = len(cancer_gene_list)

results = np.zeros((n_genes, n_runs))

low_vec = start_end[0]
high_vec = start_end[1]

lo = np.minimum(low_vec, high_vec)
hi = np.maximum(low_vec, high_vec)

np.random.seed(1)

for i in range(n_runs):
    rand_exp = np.random.uniform(lo, hi)

    df_state = NetDesDuo.dynamic_state(
        dt=1,
        t_tot=2000,
        exp_array=rand_exp,
        parameters_all=cancer_res_final,
        gene_position=gene_position2,
        e_mean=mean_exp2,
        save_num=201
    )

    results[:, i] = df_state.iloc[:, -1].values


cancer_baseline_cloud = pd.DataFrame(
    results,
    index=cancer_gene_list,
    columns=[f"run_{j+1}" for j in range(n_runs)]
)
cancer_baseline_cloud.to_csv("cancer_perturbations/cancer_baseline_cloud.csv")

In [ ]:
# -----------------------------
# Cancer baseline cloud + full real pseudotime trajectory
# -----------------------------
cancer_baseline_cloud = pd.read_csv("cancer_perturbations/cancer_baseline_cloud.csv", index_col = 0)

# baseline cloud: runs x genes
X_cloud = cancer_baseline_cloud.T.values

# fit PCA on the simulated baseline cloud
pca_model = PCA(n_components=2, random_state=0)
pcs_cloud = pca_model.fit_transform(X_cloud)

explained = pca_model.explained_variance_ratio_ * 100

pcs_cloud_df = pd.DataFrame(
    pcs_cloud,
    index=cancer_baseline_cloud.columns,
    columns=["PC1", "PC2"]
)

# real pseudotime data in same scaled expression space
cancer_scaled = cancer_expression.div(cancer_expression.max(axis=1), axis=0)
cancer_scaled = cancer_scaled.loc[cancer_gene_list]

# pseudotime points: timepoints x genes
X_real = cancer_scaled.T.values

# project real pseudotime trajectory into the baseline PCA axes
pcs_real = pca_model.transform(X_real)

pcs_real_df = pd.DataFrame(
    pcs_real,
    columns=["PC1", "PC2"]
)

pcs_real_df["pseudotime_index"] = np.arange(len(pcs_real_df))

# plot
plt.figure(figsize=(6, 5))

plt.scatter(
    pcs_cloud_df["PC1"],
    pcs_cloud_df["PC2"],
    s=12,
    alpha=0.35,
    label="Simulated States"
)


#plt.scatter(
#    pcs_real_df["PC1"],
#    pcs_real_df["PC2"],
#    c=pcs_real_df["pseudotime_index"],
#    s=25,
#    cmap="viridis",
#    label="Pseudotime Points"
#)

plt.scatter(
    pcs_real_df["PC1"].iloc[0],
    pcs_real_df["PC2"].iloc[0],
    s=120,
    marker="o",
    edgecolor="black",
    linewidth=1.2,
    label="Observed Start"
)

plt.scatter(
    pcs_real_df["PC1"].iloc[-1],
    pcs_real_df["PC2"].iloc[-1],
    s=120,
    marker="s",
    edgecolor="black",
    linewidth=1.2,
    label="Observed End"
)


plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.xlabel(f"PC1 ({explained[0]:.1f}%)", fontsize = 14)
plt.ylabel(f"PC2 ({explained[1]:.1f}%)", fontsize = 14)
#plt.colorbar(label="Pseudotime index")
plt.legend(fontsize=10)

plt.savefig('cancer_results/cancer_stochastic_simulation.png', dpi = 600)

plt.tight_layout()
plt.show()

## Perturbation: force and release

In [ ]:
naive_baseline_cloud = pd.read_csv("naive_perturbations/naive_baseline_cloud.csv", index_col = 0)
naive_scaled = naive_expression.div(naive_expression.max(axis=1), axis=0)

#min to max for that gene
naive_start_end = [
    naive_scaled.min(axis=1).values.astype(float), 
    naive_scaled.max(axis=1).values.astype(float), 
]

naive_gene_to_i = {g: i for i, g in enumerate(naive_gene_list)}

naive_gene_position2 = [
    [naive_gene_to_i[r] for r in regs]
    for regs in naive_ob_newall
]

naive_e_mean = naive_scaled.mean(axis=1).values.astype(float)

per = 0.3
n_runs = 2000
L = len(naive_gene_list)

naive_pair_results = {}

print('here')
results = joblib.Parallel(n_jobs=20, verbose = 100, backend="loky")(
    joblib.delayed(NetDesDuo.run_one_pair)(i, j, per, n_runs, naive_baseline_cloud, naive_start_end,
                                 naive_gene_list, naive_res_final, naive_gene_position2,
                                 naive_e_mean, "naive_perturbations/naive_perturbed")
    for i in range(L) for j in range(i, L))

In [ ]:
cancer_baseline_cloud = pd.read_csv("cancer_perturbations/cancer_baseline_cloud.csv", index_col = 0)
cancer_scaled = cancer_expression.div(cancer_expression.max(axis=1), axis=0)

#min to max for that gene
cancer_start_end = [
    cancer_scaled.min(axis=1).values.astype(float), 
    cancer_scaled.max(axis=1).values.astype(float), 
]

cancer_gene_to_i = {g: i for i, g in enumerate(cancer_gene_list)}

cancer_gene_position2 = [
    [cancer_gene_to_i[r] for r in regs]
    for regs in cancer_ob_newall
]

cancer_e_mean = cancer_scaled.mean(axis=1).values.astype(float)

per = 0.3
n_runs = 2000
L = len(cancer_gene_list)

cancer_pair_results = {}

results = joblib.Parallel(n_jobs=20, verbose = 100,  backend="loky")(
    joblib.delayed(NetDesDuo.run_one_pair)(i, j, per, n_runs, cancer_baseline_cloud, cancer_start_end,
                                 cancer_gene_list, cancer_res_final, cancer_gene_position2,
                                 cancer_e_mean, "cancer_perturbations/cancer_perturbed")
    for i in range(L) for j in range(i, L))

## Perturbation-strength sweep (figure 6B)

In [ ]:
#varying value of per for most positive + most negative pairs in naive and cancer
naive_baseline_cloud = pd.read_csv("naive_perturbations/naive_baseline_cloud.csv", index_col=0)
naive_scaled = naive_expression.div(naive_expression.max(axis=1), axis=0)

#min to max for that gene
naive_start_end = [
    naive_scaled.min(axis=1).values.astype(float), 
    naive_scaled.max(axis=1).values.astype(float), 
]

naive_gene_to_i = {g: i for i, g in enumerate(naive_gene_list)}
naive_gene_position2 = [
    [naive_gene_to_i[r] for r in regs]
    for regs in naive_ob_newall
]

naive_e_mean = naive_scaled.mean(axis=1).values.astype(float)

#change these
naive_pairs = [["Egr1", "Cebpb"], ["Egr1", "Nfkb2"], ["Ets1", "Ets1"], ["Ets1", "Fos"]]
n_runs = 2000
per_grid = [i/10 for i in range(0, 11)]

results = joblib.Parallel(n_jobs=22, verbose = 100, backend="loky")(
    joblib.delayed(NetDesDuo.run_one_sweep)(
        pair, per, n_runs, naive_baseline_cloud, naive_start_end, naive_gene_list,
        naive_res_final, naive_gene_position2, naive_e_mean,
        f"naive_perturbations/naive_per_sweep/naive_{pair[0]}_{pair[1]}_{per}.csv",
        seed=0, release_t_tot=2000)
    for pair in naive_pairs for per in per_grid)

In [ ]:
cancer_baseline_cloud = pd.read_csv("cancer_perturbations/cancer_baseline_cloud.csv", index_col = 0)
cancer_scaled = cancer_expression.div(cancer_expression.max(axis=1), axis=0)

#min and max of each gene
cancer_start_end = [
    cancer_scaled.min(axis=1).values.astype(float),
    cancer_scaled.max(axis=1).values.astype(float),
]

cancer_gene_to_i = {g: i for i, g in enumerate(cancer_gene_list)}

cancer_gene_position2 = [
    [cancer_gene_to_i[r] for r in regs]
    for regs in cancer_ob_newall
]

cancer_e_mean = cancer_scaled.mean(axis=1).values.astype(float)

#change these 
cancer_pairs = [["Cebpb", "Jun"], ["Fos", "Egr1"], ["Ets1", "Stat1"], ["Ets1", "Ets1"]]
n_runs = 2000
per_grid = [i/10 for i in range(0, 11)]

results = joblib.Parallel(n_jobs=22, verbose = 1000, backend="loky")(
    joblib.delayed(NetDesDuo.run_one_sweep)(
        pair, per, n_runs, cancer_baseline_cloud, cancer_start_end, cancer_gene_list,
        cancer_res_final, cancer_gene_position2, cancer_e_mean,
        f"cancer_perturbations/cancer_per_sweep/cancer_{pair[0]}_{pair[1]}_{per}.csv",
        seed=0, release_t_tot=2000)
    for pair in cancer_pairs for per in per_grid)